# 技能5 · Day 3 上机：用 deepeval 搭建营销 Agent 评测套件

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **deepeval** 为营销 Agent 搭建可运行的评测套件
2. 区分**轨迹评估**（工具调用正确性）与**端到端评估**（内容质量），分别用自定义 BaseMetric 和 GEval 实现
3. 用 **FaithfulnessMetric** 检测幻觉（输出是否忠于知识库），用 **LLM-as-a-judge** 自动评审轨迹质量
4. 用 `evaluate()` 批量运行测试套件，计算任务完成率/工具调用准确率/幻觉率

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：deepeval（confident-ai/deepeval，17k★，LLM评估框架）。
营销映射：评估营销内容生成Agent的轨迹质量（是否选对工具、是否生成合规内容）。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> ⚠️ deepeval 默认使用 OpenAI 作为 judge 模型，需设置 `OPENAI_API_KEY` 环境变量。
> 也可配置其他模型（如 Anthropic / 本地 Ollama），见 [deepeval 文档](https://docs.confident-ai.com/)。

In [ ]:
# !pip install deepeval -q
# export OPENAI_API_KEY=<your-openai-api-key>

## 1. 数据集背景与营销映射

**评估对象**：营销内容生成 Agent 的真实输出轨迹。我们定义3个测试用例，分别代表好/坏/混合轨迹：

| 用例 | 场景 | 轨迹质量 | 评估重点 |
|------|------|---------|---------|
| 用例1 | 小红书种草文案（烟酰胺精华液） | 好（工具正确、内容忠于知识库） | 端到端质量 + 幻觉检测应通过 |
| 用例2 | 朋友圈广告（新款丝绒口红） | 差（跳过搜索、虚构成分） | 工具调用准确率 + 幻觉率应报警 |
| 用例3 | 小红书种草文案（防晒霜） | 混合（工具正确但内容略有偏差） | 轨迹评估 vs 端到端评估的差异 |

每条测试用例包含：
- `input`：营销 Brief（产品+目标人群+渠道）
- `actual_output`：Agent 实际生成的内容
- `expected_output`：人工专家写的参考文案
- `retrieval_context`：知识库检索到的产品资料（用于幻觉检测）
- `trajectory`：Agent 的工具调用轨迹（用于轨迹评估，自定义属性）

**营销映射**：在真实项目中，这些数据来自你的 Agent 的实际运行日志。本上机用预置的真实场景数据。

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from deepeval import assert_test, evaluate
from deepeval.metrics import GEval, BaseMetric, FaithfulnessMetric
from deepeval.test_case import LLMTestCase

# deepeval 新版将 LLMTestCaseParams 重命名为 SingleTurnParams，此处兼容两种版本
try:
    from deepeval.test_case import LLMTestCaseParams
except ImportError:
    from deepeval.test_case import SingleTurnParams as LLMTestCaseParams

print("deepeval 导入完成")

## TODO 1：准备营销Agent真实轨迹测试数据

In [ ]:
# TODO 1：准备营销 Agent 真实轨迹测试数据
# 提示：用 LLMTestCase 定义3个测试用例（好/坏/混合轨迹）
#   test_case = LLMTestCase(input="...", actual_output="...",
#                           expected_output="...", retrieval_context=[...])
#   test_case.trajectory = [{"tool": "...", "params": {...}, "correct": True}, ...]
# 要求：定义3个测试用例，分别代表好/坏/混合轨迹

# ===== 你的代码 =====
test_case_good = None   # TODO: 好轨迹（工具正确、内容忠于知识库）
test_case_bad = None    # TODO: 坏轨迹（跳过搜索、虚构成分）
test_case_mixed = None  # TODO: 混合轨迹（工具正确但内容略有偏差）
# ====================

print(f"用例1（好轨迹）输出长度: {len(test_case_good.actual_output)} 字")
print(f"用例2（坏轨迹）输出长度: {len(test_case_bad.actual_output)} 字")
print(f"用例3（混合轨迹）输出长度: {len(test_case_mixed.actual_output)} 字")

## 2. 轨迹评估 vs 端到端评估

Agent 评估的核心洞察：**不能只看答案对不对，必须评估过程好不好**。

```
端到端评估（End-to-End）：
  Brief -> [Agent 黑盒] -> 文案 -> 对比参考文案 -> 好/差

轨迹评估（Trajectory）：
  Brief -> Thought -> Action(搜索知识库) -> Obs -> Thought -> Action(生成文案) -> 文案
                 ↑                    ↑                              ↑
           推理合理？           工具选对？参数对？                步骤冗余？
```

- **端到端**：用 GEval 让 LLM-as-judge 评估内容质量（品牌调性、CTA、平台适配）--粗粒度
- **轨迹**：用自定义 BaseMetric 评估工具调用正确性（选对工具？参数正确？）--细粒度
- **幻觉**：用 FaithfulnessMetric 对比 actual_output 与 retrieval_context --检测虚构成分

两层都要：端到端做基础门禁，轨迹做深度诊断。

## TODO 2-3：端到端评估 + 轨迹评估

In [ ]:
# TODO 2：端到端评估 -- 用 GEval 评估营销内容质量
# 提示：GEval(name=..., criteria=..., evaluation_params=[...], threshold=...)
#   evaluation_params 用 LLMTestCaseParams.INPUT, .ACTUAL_OUTPUT, .EXPECTED_OUTPUT
#   criteria 描述评估标准（品牌调性/CTA/平台适配/情感共鸣）
# 要求：定义GEval指标，对3个测试用例分别 measure，打印评分和理由

# ===== 你的代码 =====
content_quality_metric = None  # TODO: 定义 GEval 指标
# ====================

for i, tc in enumerate([test_case_good, test_case_bad, test_case_mixed], 1):
    content_quality_metric.measure(tc)
    print(f"用例{i} 内容质量: {content_quality_metric.score:.2f} | {content_quality_metric.reason[:100]}")

In [ ]:
# TODO 3：轨迹评估 -- 自定义 BaseMetric 评估工具调用正确性
# 提示：继承 BaseMetric，实现 measure / a_measure / is_successful / __name__
#   measure 中读取 test_case.trajectory，计算正确调用比例
#   trajectory 格式: [{"tool": "...", "params": {...}, "correct": True/False}, ...]
# 要求：实现 ToolCallAccuracyMetric，对3个用例 measure，打印评分和理由

# ===== 你的代码 =====
class ToolCallAccuracyMetric(BaseMetric):
    def __init__(self, threshold=0.7):
        self.threshold = threshold

    # TODO: 实现 measure, a_measure, is_successful, __name__
    pass
# ====================

tool_metric = ToolCallAccuracyMetric()
for i, tc in enumerate([test_case_good, test_case_bad, test_case_mixed], 1):
    tool_metric.measure(tc)
    print(f"用例{i} 工具调用准确率: {tool_metric.score:.2f} | {tool_metric.reason}")

## 3. 幻觉检测：为什么营销 Agent 必须做

营销 Agent 的幻觉后果严重：
- **虚构成分/功效**：如把"哑光口红"说成"含玻尿酸滋润"（产品实际不含）-> 违反广告法
- **虚构价格/优惠**：如编造"限时5折"（实际无此活动）-> 虚假宣传
- **虚构认证**：如编造"皮肤科医生推荐"（实际无此背书）-> 误导消费者

**FaithfulnessMetric 的工作原理**：
1. 从 `actual_output` 中提取所有事实性声明（claims）
2. 逐条与 `retrieval_context`（知识库）交叉验证
3. 忠实度 = 忠实声明数 / 总声明数
4. 低于 threshold 的用例 = 存在幻觉

这是 deepeval 内置指标，无需手写规则匹配。

## TODO 4-5：幻觉检测 + LLM-as-a-judge自动评审

In [ ]:
# TODO 4：幻觉检测 -- 用 FaithfulnessMetric 检测输出是否忠于知识库
# 提示：FaithfulnessMetric(threshold=0.7, include_reason=True)
#   它会对比 actual_output 与 retrieval_context，检测每条声明是否忠于知识库
#   忠实度 < threshold = 存在幻觉
# 要求：对3个用例 measure，打印忠实度评分和理由

# ===== 你的代码 =====
faithfulness_metric = None  # TODO: 定义 FaithfulnessMetric
# ====================

for i, tc in enumerate([test_case_good, test_case_bad, test_case_mixed], 1):
    faithfulness_metric.measure(tc)
    print(f"用例{i} 忠实度: {faithfulness_metric.score:.2f} | {faithfulness_metric.reason[:100]}")

In [ ]:
# TODO 5：LLM-as-a-judge 自动评审 -- 用 GEval criteria 评估轨迹质量
# 提示：GEval(name="轨迹质量", criteria="评估工具选择/推理链/冗余步骤/参数准确",
#   evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT])
# 注意：这是 LLM-as-a-judge 范式（arXiv 2306.05685），用LLM自动评估Agent输出质量
# 要求：定义指标，对3个用例 measure，打印评分和理由

# ===== 你的代码 =====
trajectory_judge_metric = None  # TODO: 定义 GEval（LLM-as-a-judge）
# ====================

for i, tc in enumerate([test_case_good, test_case_bad, test_case_mixed], 1):
    trajectory_judge_metric.measure(tc)
    print(f"用例{i} 轨迹质量(LLM-judge): {trajectory_judge_metric.score:.2f} | {trajectory_judge_metric.reason[:100]}")

## 5. 综合评估指标

完成 TODO 1-5 后，我们有了3个维度 × 3个用例的评估结果。TODO 6 将用 `evaluate()` 批量运行，并汇总为三个核心指标：

| 指标 | 定义 | 计算方式 | 营销 Agent 目标 |
|------|------|---------|----------------|
| 任务完成率 | 内容质量达标的用例比例 | GEval score ≥ threshold 的用例数 / 总用例数 | ≥ 85% |
| 工具调用准确率 | 工具调用正确的比例 | ToolCallAccuracyMetric score 的均值 | ≥ 90% |
| 幻觉率 | 存在幻觉的用例比例 | FaithfulnessMetric score < threshold 的用例数 / 总用例数 | ≤ 5% |

`evaluate()` 会批量运行所有 test_case × metric 组合，返回结构化结果。

## TODO 6：综合评估（evaluate批量运行）

In [ ]:
# TODO 6：综合评估 -- 用 evaluate 批量运行，计算任务完成率/工具准确率/幻觉率
# 提示：evaluate(test_cases=[...], metrics=[...]) 返回结构化结果
#   然后从各 metric 的 score 手动汇总三个指标：
#   - 任务完成率 = content_quality score >= 0.7 的用例比例
#   - 工具调用准确率 = tool_call score 的均值
#   - 幻觉率 = faithfulness score < 0.7 的用例比例
# 要求：运行 evaluate，打印三个综合指标

# ===== 你的代码 =====
results = None  # TODO: 用 evaluate 批量运行
# 计算三个综合指标
task_completion_rate = None   # TODO: 内容质量 >= 0.7 的比例
tool_accuracy_rate = None     # TODO: 工具调用准确率的均值
hallucination_rate = None     # TODO: 忠实度 < 0.7 的比例（即幻觉率）
# ====================

print("=" * 50)
print(f"任务完成率: {task_completion_rate:.1%}")
print(f"工具调用准确率: {tool_accuracy_rate:.1%}")
print(f"幻觉率: {hallucination_rate:.1%}")
print("=" * 50)

## 6. 反思与前沿

### 反思问题
1. 你的营销 Agent 在哪个评估维度表现最差？根因是什么（工具选择/参数/推理/幻觉）？
2. 用例2（坏轨迹）的工具调用准确率为0%，但端到端内容质量评分可能不是0--为什么？（提示：LLM 可能生成"看起来合理"但实际有幻觉的内容）
3. 如果 Agent 在测试集上表现好，但在生产中表现差，可能是什么原因？（提示：测试集覆盖不足/长尾问题/分布漂移）
4. LLM-as-a-judge 的评分本身是否可信？如何校准？（提示：人工抽检 + 多 judge 投票）

### 2026 前沿：LLM-as-a-judge + deepeval 可运行评测框架
把 LLM-as-a-judge（NeurIPS 2023, arXiv 2306.05685）写成 deepeval 的 GEval 测试用例，用 `assert_test` 断言 + `deepeval test run` 在 CI 中自动执行：
- 每次模型升级/prompt修改后，自动运行完整测试集
- 评分低于 threshold 的用例自动 fail，防止回归
- LLM-as-judge 的评分+理由结构化存储，支持按时间/场景聚合分析

**注意**：LLM-as-a-judge 是辅助评估，有自身偏差。对应因果阶梯 L1（对轨迹文本的关联分析），不能替代真实业务指标（L2 A/B测试）。定位为"开发期自检工具"。

参考 [arXiv 2306.05685](https://arxiv.org/abs/2306.05685)（NeurIPS 2023, LLM-as-a-judge）+ [deepeval](https://github.com/confident-ai/deepeval)。